In [1]:
import pandas as pd
import os

In [3]:
from pathlib import Path
ROOT = Path.cwd().parent

In [4]:
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"

In [5]:
# Create processed directory
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
# Load raw data
def load_raw():

    sales = pd.read_csv(RAW_DIR / "sales_daily.csv")
    sku = pd.read_csv(RAW_DIR / "sku_master.csv")
    calendar = pd.read_csv(RAW_DIR / "calendar.csv")
    inventory = pd.read_csv(RAW_DIR / "inventory_snapshots.csv")

    # Remove leading/trailing spaces from column names
    sales.columns = sales.columns.str.strip()
    sku.columns = sku.columns.str.strip()
    calendar.columns = calendar.columns.str.strip()
    inventory.columns = inventory.columns.str.strip()

    return sales, sku, calendar, inventory

In [8]:
def sales_clean(sales):
    sales = sales.copy()
    sales["Date"] = pd.to_datetime(sales["Date"])
    sales = sales.drop_duplicates(subset=["Date", "SKU"])

    sales = sales[
        (sales["Units_Sold"] >= 0) &
        (sales["Revenue"] >= 0)
    ]

    sales = sales.rename(columns={
        "Date": "date",
        "SKU": "sku_id",
        "Units_Sold": "units_sold",
        "Revenue": "revenue",
        "Price": "unit_price",
        "Promotion": "promo_flag"
    })

    return sales

In [9]:
def sku_master_clean(sku):

    sku = sku.copy()
    sku["Launch_Date"] = pd.to_datetime(sku["Launch_Date"])
    sku = sku.drop_duplicates(subset=["SKU"])

    sku = sku.rename(columns={
        "SKU": "sku_id",
        "Product_Name": "product_name",
        "Category": "category",
        "Subcategory": "subcategory",
        "Launch_Date": "launch_date",
        "Cost_Price": "unit_cost",
        "Selling_Price": "list_price",
        "Gross_Margin_Per_Unit": "gross_margin_per_unit"
    })

    sku["is_loss_making"] = (
        sku["gross_margin_per_unit"] < 0
    )

    return sku

In [10]:
def calendar_clean(calendar):

    calendar = calendar.copy()
    calendar["date"] = pd.to_datetime(calendar["date"])
    calendar = calendar.drop_duplicates(subset=["date"])

    calendar["holiday"] = (
        calendar["holiday"]
        .fillna("No Holiday")
        .astype(str)
        .str.strip()
    )

    calendar["promotion_event"] = (
        calendar["promotion_event"]
        .fillna("No Promotion")
        .astype(str)
        .str.strip()
    )

    return calendar

In [12]:
def inventory_clean(inventory):

    inventory = inventory.copy()
    inventory["Snapshot_Date"] = pd.to_datetime(inventory["Snapshot_Date"])
    inventory = inventory.drop_duplicates(subset=["Snapshot_Date", "SKU"])

    inventory = inventory.rename(columns={
        "Snapshot_Date": "date",
        "SKU": "sku_id",
        "Current_Stock": "on_hand_units",
        "On_Order": "on_order_units",
        "Lead_Time_Days": "lead_time_days",
        "Safety_Stock": "safety_stock",
        "Reorder_Point": "reorder_point",
        "Inventory_Value": "inventory_value"
    })

    return inventory

In [13]:
def build_master_dataset(sales, sku, cal):
    df = sales.merge(sku, on="sku_id", how="left")
    df = df.merge(cal, on="date", how="left")
    return df

In [14]:
def run_pipeline():
    sales_raw, sku_raw, cal_raw, inv_raw = load_raw()

    sales = sales_clean(sales_raw)
    sku = sku_master_clean(sku_raw)
    cal = calendar_clean(cal_raw)
    inv = inventory_clean(inv_raw)

    skus_with_sales = set(sku["sku_id"])
    inv["has_sales_history"] = inv["sku_id"].isin(skus_with_sales)

    master = build_master_dataset(sales, sku, cal)

    master.to_csv(PROCESSED_DIR / "master_sales.csv",index=False)
    inv.to_csv(PROCESSED_DIR / "inventory_clean.csv",index=False)
    sku.to_csv(PROCESSED_DIR / "sku_master_clean.csv",index=False)

    print(f"master_sales.csv: {master.shape}")
    print(f"inventory_clean.csv: {inv.shape}")
    print(f"sku_master_clean.csv: {sku.shape}")
    print(f"SKUs with sales history: {len(skus_with_sales)} / {inv['sku_id'].nunique()} in inventory")
    print(f"Loss-making SKUs flagged: {sku['is_loss_making'].sum()}")

In [15]:
run_pipeline()

master_sales.csv: (36550, 24)
inventory_clean.csv: (4800, 9)
sku_master_clean.csv: (50, 9)
SKUs with sales history: 50 / 200 in inventory
Loss-making SKUs flagged: 16
